# Model fixation analysis

Publication metadata

- **Title:** Model fixation analysis for offspring-distribution comparisons
- **Version:** 1.0.0
- **Date:** 2026-05-26

This notebook estimates the fate of a single beneficial allele lineage under Poisson, negative binomial, and fractional offspring distributions. The simulation starts with one heterozygous individual carrying allele `A` on an ancestral `aa` background. Because the model follows the early fate of a rare allele in an effectively infinite population, `AA` homozygotes are not produced and the resident background is not simulated explicitly.

Each copy of allele `A` has fitness `1 + s` relative to the ancestral genotype. The model is agnostic about whether this advantage reflects viability before breeding or fecundity among surviving individuals; in either case, selection changes the expected reproductive output of the `A` lineage. Runs continue until the allele is lost (`0` copies) or reaches the copy-number threshold.

Throughout the notebook, a replicate means one copy of allele `A` in the current generation. Family-size variance and effective population size are calculated from the neutral offspring distribution before introducing the beneficial allele, so those quantities use `s = 0` even when the fixation simulation uses `s > 0`.


## Helper Functions and Setup

This cell defines configuration, sampling functions, analytical variance calculations, theoretical fixation calculations, and simulation routines used by the rest of the notebook. Runtime parameters can be overridden with environment variables; see `README.md` for the list.

The sampling functions are:

- `draw_offspring(..., dist="poisson")`: Poisson offspring with mean `mu = 1 + s` per `A` copy.
- `draw_offspring(..., dist="negative_binomial")`: negative-binomial offspring with mean `mu = 1 + s` and dispersion controlled by `r`. In the neutral case used for variance calculations, per-copy offspring variance is `1 + 2/r`; family-size variance is twice that value, `2 + 4/r`.
- `draw_offspring(..., dist="fractional")`: each copy has probability `f` of producing zero offspring; the fractional mean is set to `mu / (1 - f)` so the overall mean remains `mu = 1 + s`. For the fractional draw, `E[X] = (1 - f) mu_frac` and `Var(X) = (1 - f) mu_frac (1 + f mu_frac)`. After density dependence resets the neutral mean to `1`, this gives per-copy variance `1/(1-f)` and family-size variance `2/(1-f)`.


In [ ]:
# Title: Model fixation analysis helpers
# Description: Configuration, theory helpers, and simulation functions for Model.ipynb.
# Version: 1.0.0
# Date: 2026-05-26

import logging
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd

log = logging.getLogger("model")
if not log.handlers:
    logging.basicConfig(level=os.getenv("MODEL_LOG_LEVEL", "INFO"), format="%(message)s")


@dataclass(frozen=True)
class ModelConfig:
    s: float = 0.02
    threshold: int = 250
    n_census: int = 200_000
    initial: int = 1
    max_gen: int = 100_000
    sims: int = 10_000
    reps: int = 100
    frac_zeros: tuple[float, ...] = (0.4, 0.8)
    r_nb: float = 2.0
    seed: int = 12345

    @property
    def p_nb(self) -> float:
        return 2.0 / (self.r_nb + 2.0)


def _env_int(name, default):
    val = os.getenv(name)
    if val in (None, ""):
        return default
    try:
        return int(val)
    except ValueError:
        log.warning("Ignoring invalid %s=%r", name, val)
        return default


def _env_float(name, default):
    val = os.getenv(name)
    if val in (None, ""):
        return default
    try:
        return float(val)
    except ValueError:
        log.warning("Ignoring invalid %s=%r", name, val)
        return default


def config_from_env() -> ModelConfig:
    frac = os.getenv("MODEL_FRAC_ZEROS")
    if frac:
        try:
            frac_zeros = tuple(float(x.strip()) for x in frac.split(",") if x.strip())
        except ValueError:
            log.warning("Ignoring invalid MODEL_FRAC_ZEROS=%r", frac)
            frac_zeros = (0.4, 0.8)
    else:
        frac_zeros = (0.4, 0.8)

    return ModelConfig(
        s=_env_float("MODEL_S", 0.02),
        threshold=_env_int("MODEL_THRESHOLD", 250),
        n_census=_env_int("MODEL_N_CENSUS", 200_000),
        initial=_env_int("MODEL_INITIAL", 1),
        max_gen=_env_int("MODEL_MAX_GEN", 100_000),
        sims=_env_int("MODEL_SIMS", 10_000),
        reps=_env_int("MODEL_REPS", 100),
        frac_zeros=frac_zeros,
        r_nb=_env_float("MODEL_R_NB", 2.0),
        seed=_env_int("MODEL_SEED", 12345),
    )


def require_probability(x, name):
    if x is None or not 0 <= x < 1:
        raise ValueError(f"{name} must be in [0, 1).")


def fractional_moments(fractional_mean, frac_zero):
    """Mean and variance for the fractional offspring draw."""
    require_probability(frac_zero, "frac_zero")
    mean = (1.0 - frac_zero) * fractional_mean
    var = (1.0 - frac_zero) * fractional_mean * (1.0 + frac_zero * fractional_mean)
    return mean, var


def neutral_family_variance(dist, *, f=None, r=None):
    """Analytical family-size variance before introducing the selected allele."""
    if dist == "poisson":
        return 2.0
    if dist == "negative_binomial":
        if r is None or r <= 0:
            raise ValueError("r must be positive for negative_binomial.")
        return 2.0 + 4.0 / r
    if dist == "fractional":
        fractional_mean = 1.0 / (1.0 - f)
        _, var_copy = fractional_moments(fractional_mean, f)
        return 2.0 * var_copy
    raise ValueError(f"Unknown distribution: {dist}")


def effective_size_from_family_variance(vk, n_census):
    """Convert neutral family-size variance to effective population size."""
    return ((4.0 * n_census) - 4.0) / (vk + 2.0)


def kimura_fixation_probability(s, ne, n_census):
    """Large-N Kimura approximation used for the negative-binomial sweep."""
    return -np.expm1(-2.0 * s * (ne / n_census))


def branching_fixation_probability(dist, s, *, f=None, p_nb=None, tol=1e-14, max_iter=100_000):
    mu = 1.0 + s

    # Survival probability is 1 - q, where q is the smallest extinction fixed point.

    if dist == "poisson":
        pgf = lambda q: np.exp(mu * (q - 1.0))
    elif dist == "negative_binomial":
        require_probability(p_nb, "p_nb")
        raw_mu = mu / (1.0 - p_nb)
        var = (1.0 - p_nb) * raw_mu * (1.0 + p_nb * raw_mu)
        size = mu**2 / (var - mu)
        prob = size / (size + mu)
        pgf = lambda q: (prob / (1.0 - (1.0 - prob) * q)) ** size
    elif dist == "fractional":
        require_probability(f, "frac_zero")
        fractional_mean = mu / (1.0 - f)
        pgf = lambda q: f + (1.0 - f) * np.exp(fractional_mean * (q - 1.0))
    else:
        raise ValueError(f"Unknown distribution: {dist}")

    q = 0.0
    for _ in range(max_iter):
        nxt = float(pgf(q))
        if abs(nxt - q) < tol:
            return float(np.clip(1.0 - nxt, 0.0, 1.0))
        q = nxt
    log.warning("Extinction fixed-point iteration did not converge for %s", dist)
    return float(np.clip(1.0 - q, 0.0, 1.0))


def draw_offspring(rng, copies, dist, mu, *, f=None, p_nb=None):
    """Draw the next generation of A-allele copies for the chosen offspring model."""
    copies = copies.astype(float, copy=False)

    if dist == "poisson":
        return rng.poisson(mu * copies)

    if dist == "negative_binomial":
        require_probability(p_nb, "p_nb")
        raw_mu = mu / (1.0 - p_nb)
        var_copy = (1.0 - p_nb) * raw_mu * (1.0 + p_nb * raw_mu)
        mean = mu * copies
        var = var_copy * copies
        if np.isclose(var_copy, mu):
            return rng.poisson(mean)
        size = mean**2 / (var - mean)
        prob = size / (size + mean)
        return rng.negative_binomial(size, prob)

    if dist == "fractional":
        require_probability(f, "frac_zero")
        active = rng.binomial(copies.astype(int), 1.0 - f)
        fractional_mean = mu / (1.0 - f)
        return rng.poisson(fractional_mean * active)

    raise ValueError(f"Unknown distribution: {dist}")


def run_replicate_batch(rng, dist, cfg, *, f=None, p_nb=None):
    mu = 1.0 + cfg.s
    copies = np.full(cfg.sims, cfg.initial, dtype=int)
    active_runs = np.ones(cfg.sims, dtype=bool)
    fixed = np.zeros(cfg.sims, dtype=bool)
    gen1_loss = np.nan

    for gen in range(1, cfg.max_gen + 1):
        if not active_runs.any():
            return fixed.mean(), gen - 1, gen1_loss

        idx = np.flatnonzero(active_runs)
        nxt = draw_offspring(rng, copies[idx], dist, mu, f=f, p_nb=p_nb)
        copies[idx] = nxt

        lost = nxt == 0
        if gen == 1:
            gen1_loss = float(lost.mean())
        hit = nxt >= cfg.threshold
        fixed[idx[hit]] = True
        active_runs[idx[lost | hit]] = False

    raise RuntimeError(f"{active_runs.sum()} {dist} runs unresolved after {cfg.max_gen:,} generations")


def summarize_model(cfg):
    rng = np.random.default_rng(cfg.seed)
    specs = [
        ("poisson", None, None),
        ("negative_binomial", None, cfg.r_nb),
        *[("fractional", f, None) for f in cfg.frac_zeros],
    ]
    rows = []

    for dist, f, r in specs:
        pfix_values, stop_generations, first_gen_losses = [], [], []
        for _ in range(cfg.reps):
            pfix, last_gen, first_loss = run_replicate_batch(
                rng,
                dist,
                cfg,
                f=f,
                p_nb=cfg.p_nb if dist == "negative_binomial" else None,
            )
            pfix_values.append(pfix)
            stop_generations.append(last_gen)
            first_gen_losses.append(first_loss)

        pfix_values = np.asarray(pfix_values)
        vk = neutral_family_variance(dist, f=f, r=r)
        ne = effective_size_from_family_variance(vk, cfg.n_census)
        rows.append({
            "selection": cfg.s,
            "distribution": dist,
            "threshold_copies": cfg.threshold,
            "frac_zero": f,
            "r": r,
            "n": cfg.reps,
            "mean_pfix": float(pfix_values.mean()),
            "theoretical_pfix": branching_fixation_probability(
                dist,
                cfg.s,
                f=f,
                p_nb=cfg.p_nb if dist == "negative_binomial" else None,
            ),
            "sd_pfix": float(pfix_values.std(ddof=1)) if len(pfix_values) > 1 else 0.0,
            "first_gen_loss_freq": float(np.nanmean(first_gen_losses)) if dist == "fractional" else np.nan,
            "variance": vk,
            "N_census": cfg.n_census,
            "Ne": ne,
            "mean_generations": float(np.mean(stop_generations)),
            "max_generations_observed": int(np.max(stop_generations)),
        })
    return pd.DataFrame(rows)


## Model Inputs and Run

This cell builds the run configuration, executes the model, and displays the CSV-ready summary table. The default run uses `s = 0.02`, two fractional settings (`f = 0.4` and `f = 0.8`), a negative-binomial exemplar with `r = 2`, `100` replicate batches, and `10,000` simulations per batch. Each simulation stops at loss or threshold fixation, with `MODEL_MAX_GEN` as a fail-safe rather than a fixed stopping time.


In [ ]:
cfg = config_from_env()
model_summary = summarize_model(cfg)

cols = [
    "selection",
    "distribution",
    "threshold_copies",
    "frac_zero",
    "r",
    "n",
    "mean_pfix",
    "theoretical_pfix",
    "sd_pfix",
    "first_gen_loss_freq",
    "variance",
    "N_census",
    "Ne",
]

model_summary = model_summary[cols]
model_summary_display = model_summary.copy()
def format_optional_number(value):
    if pd.isna(value):
        return ""
    return int(value) if float(value).is_integer() else value


model_summary_display["frac_zero"] = model_summary_display["frac_zero"].apply(format_optional_number)
model_summary_display["r"] = model_summary_display["r"].apply(format_optional_number)

model_summary_display


## CSV Output and Figures

This cell writes the model results to CSV, displays Figure 1, and recomputes the negative-binomial sweep for Figure 2.

Figure 1 compares simulated branching-process outcomes across the three offspring distributions. Its theoretical line uses the branching-process extinction fixed point for each distribution. Figure 2 is negative-binomial only and varies `r` from `0.4` to `100`, corresponding to neutral per-copy variances from `6.0` to `1.02`, neutral family-size variances from `12.0` to `2.04`, and approximate `N_e/N` ratios from `0.286` to `0.990` under the parameterization used here.


In [ ]:
# Title: Model fixation analysis output
# Description: CSV export and notebook figures for the model summaries generated above.
# Version: 1.0.0
# Date: 2026-05-26

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import ScalarFormatter

out_csv = os.getenv("MODEL_RESULTS_CSV", "model_results_summary.csv")
model_summary.to_csv(out_csv, index=False)


def label_for(row):
    dist = row["distribution"]
    if dist == "poisson":
        return "Poisson"
    if dist == "negative_binomial":
        return "Neg.\nBinomial"
    return f"Fractional\nf={row['frac_zero']:.2f}"


plot_df = model_summary.copy()
plot_df["label"] = plot_df.apply(label_for, axis=1)
labels = plot_df["label"].tolist()
x = np.arange(len(labels))

mean_pfix = plot_df["mean_pfix"].to_numpy()
theory = plot_df["theoretical_pfix"].to_numpy()
sd = plot_df["sd_pfix"].to_numpy()
ne = plot_df["Ne"].to_numpy()
ne_ratio = ne / plot_df["N_census"].to_numpy()

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
})

purple = "#7b3294"
lavender = "#c2a5cf"
light_green = "#a6dba0"
green = "#008837"
bar_edge = "#444444"
err_color = "#222222"
line_color = "#111111"
grid_color = "#E0E0E0"
palette = [purple, lavender, light_green, green]
bar_colors = [palette[i % len(palette)] for i in range(len(plot_df))]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 6.5), gridspec_kw={"height_ratios": [2.2, 1], "hspace": 0.55})

ax1.bar(x, mean_pfix, width=0.5, color=bar_colors, edgecolor=bar_edge, linewidth=0.9, zorder=3)
ax1.errorbar(x, mean_pfix, yerr=sd, fmt="none", color=err_color, capsize=4, capthick=1.0, linewidth=1.0, zorder=4)
ax1.plot(x, theory, color=line_color, linewidth=1.4, linestyle="--", marker="x", markersize=6, markeredgewidth=1.4, zorder=5)
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=9.5)
ax1.set_ylabel("Fixation probability ($p_{fix}$)", fontsize=10)
ax1.yaxis.grid(True, color=grid_color, linewidth=0.6, zorder=0)
ax1.set_axisbelow(True)
ax1.set_xlim(-0.55, len(labels) - 0.45)
ymax = max(float((mean_pfix + sd).max()), float(theory.max()))
ax1.set_ylim(0, ymax * 1.35 if ymax > 0 else 1)

sim_patch = mpatches.Patch(facecolor=purple, edgecolor=bar_edge, linewidth=0.9, label="Simulated $p_{fix}$")
theory_line = plt.Line2D([0], [0], color=line_color, linewidth=1.4, linestyle="--", marker="x", markersize=6, markeredgewidth=1.4, label="Theoretical $p_{fix}$")
err_line = plt.Line2D([0], [0], color=err_color, linewidth=1.0, label="+/- 1 SD")
ax1.legend(handles=[sim_patch, theory_line, err_line], fontsize=8.5, frameon=False, loc="upper left")
ax1.text(-0.12, 1.1, "A", transform=ax1.transAxes, fontsize=13, fontweight="bold", va="top")

for i, color in enumerate(bar_colors):
    ax2.bar(i, ne_ratio[i], width=0.5, color=color, edgecolor=bar_edge, linewidth=0.9, zorder=3)
ax2.set_xticks(x)
ax2.set_xticklabels(labels, fontsize=9.5)
ax2.set_ylabel("Effective size ratio ($N_e/N$)", fontsize=10)
ax2.yaxis.grid(True, color=grid_color, linewidth=0.6, zorder=0)
ax2.set_axisbelow(True)
ax2.set_xlim(-0.55, len(labels) - 0.45)
ax2.set_ylim(0, ne_ratio.max() * 1.2)
ax2.text(-0.12, 1.6, "B", transform=ax2.transAxes, fontsize=13, fontweight="bold", va="top")

n_cen = int(plot_df["N_census"].iloc[0])
n_reps = int(plot_df["n"].iloc[0])
r_text = int(cfg.r_nb) if float(cfg.r_nb).is_integer() else cfg.r_nb
fig.text(
    0.05,
    0,
    f"Figure 1. Fixation probability of a beneficial allele ($s$ = {cfg.s}) under offspring distributions "
    f"($N$ = {n_cen:,}, {n_reps} replicates x {cfg.sims:,} simulations each; negative binomial r = {r_text}).\n"
    "(Panel A) Bars show mean simulated $p_{fix}$ +/- 1 SD with dashed line showing branching-process theoretical $p_{fix}$. "
    "(Panel B) Effective size ratio ($N_e/N$)\n"
    "inferred from each distribution parameterisation.",
    ha="left",
    va="top",
    fontsize=8.5,
    color="#444444",
    linespacing=1.5,
)
plt.show()

# Negative-binomial sweep; recompute to avoid stale notebook state.
r_grid = [0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 50, 100]
nb_reps = _env_int("MODEL_NB_SWEEP_REPS", 15)
nb_sims = _env_int("MODEL_NB_SWEEP_SIMS", cfg.sims)
nb_gen = _env_int("MODEL_NB_SWEEP_GEN", 500)

p0 = cfg.initial / (2.0 * cfg.n_census)
threshold_freq = cfg.threshold / (2.0 * cfg.n_census)


def negative_binomial_sweep_pfix(rng, ne_val):
    eff = int(2.0 * ne_val)
    fixed = 0
    for _ in range(nb_sims):
        p = p0
        for _ in range(nb_gen):
            p_sel = (p * (1.0 + cfg.s)) / (p * (1.0 + cfg.s) + (1.0 - p))
            copies = rng.binomial(eff, p_sel)
            p = copies / eff
            if p <= 0:
                break
            if p >= threshold_freq:
                fixed += 1
                break
    return fixed / nb_sims


rng = np.random.default_rng(cfg.seed + 1)
rows = []
for r in r_grid:
    vk = neutral_family_variance("negative_binomial", r=r)
    ne_val = effective_size_from_family_variance(vk, cfg.n_census)
    vals = [negative_binomial_sweep_pfix(rng, ne_val) for _ in range(nb_reps)]
    rows.append({
        "selection": cfg.s,
        "distribution": "negative_binomial",
        "r": r,
        "n": nb_reps,
        "num_simulations": nb_sims,
        "generations": nb_gen,
        "variance": vk,
        "N_census": cfg.n_census,
        "Ne": ne_val,
        "Ne_over_N": ne_val / cfg.n_census,
        "pfix_mean": float(np.mean(vals)),
        "pfix_sd": float(np.std(vals, ddof=1)),
        "pfix_theoretical": kimura_fixation_probability(cfg.s, ne_val, cfg.n_census),
    })

nb_sweep_summary = pd.DataFrame(rows).sort_values("Ne_over_N")
xv = nb_sweep_summary["Ne_over_N"].to_numpy()
y = nb_sweep_summary["pfix_mean"].to_numpy()
y_sd = nb_sweep_summary["pfix_sd"].to_numpy()
y_theory = nb_sweep_summary["pfix_theoretical"].to_numpy()

fig2, ax = plt.subplots(figsize=(7, 4.5))
ax.fill_between(xv, np.maximum(0, y - y_sd), y + y_sd, color=lavender, edgecolor="none", zorder=2, label="+/- 1 SD")
ax.plot(xv, y, "-", color=green, linewidth=1.8, zorder=3, label="Simulated $p_{fix}$")
ax.plot(xv, y_theory, "--", color=purple, linewidth=1.5, zorder=4, label="Theoretical $p_{fix}$")
ax.set_xlabel("Effective size ratio ($N_e/N$)", fontsize=10)
ax.set_ylabel("Fixation probability ($p_{fix}$)", fontsize=10)
ax.yaxis.grid(True, color=grid_color, linewidth=0.6, zorder=0)
ax.set_axisbelow(True)
ax.set_ylim(0, max(float((y + y_sd).max()), float(y_theory.max())) * 1.12)
ax.set_xlim(max(0, float(xv.min()) - 0.01), min(1.02, float(xv.max()) + 0.01))
for axis in [ax.xaxis, ax.yaxis]:
    fmt = ScalarFormatter()
    fmt.set_scientific(False)
    fmt.set_useOffset(False)
    axis.set_major_formatter(fmt)
ax.tick_params(axis="both", labelsize=9)
ax.legend(fontsize=8.5, frameon=False, loc="upper left")
fig2.text(
    0.05,
    -0.04,
    f"Figure 2. Fixation probability of a beneficial allele ($s$ = {cfg.s}) as a function of effective size ratio "
    f"($N_e/N = 4/(V_k + 2)$)\nunder a negative binomial offspring distribution "
    f"($N$ = {cfg.n_census:,}, {nb_reps} replicates x {nb_sims:,} simulations each). "
    "Solid line: mean simulated $p_{fix}$; shaded region: +/- 1 SD;\n"
    "dashed line: Kimura large-N approximation from the effective size ratio.",
    ha="left",
    va="top",
    fontsize=8.5,
    color="#444444",
    linespacing=1.5,
)
plt.show()
